# 🐄 Notebook 2: Thundering Herd & Why Jitter Helps

Imagine 100 clients calling a service. The service blips for 1 second.
All 100 retry using *the same* exponential backoff — so they all retry **at the same moments**.
The service sees repeating spikes of 100 concurrent requests — stampede.

With **jitter**, each client waits a slightly different amount, spreading the load.

## 🛠️ Setup

```bash
cd 05-microservices/retry
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
import random, math, collections

def simulate(jitter=False, clients=100, attempts=5):
    """Count how many clients retry in each 100ms time bucket."""
    buckets = collections.Counter()
    for _ in range(clients):
        t = 0.0
        for a in range(1, attempts+1):
            delay = 0.1 * (2 ** (a-1))
            if jitter:
                delay *= random.uniform(0.5, 1.5)
            t += delay
            buckets[round(t, 1)] += 1
    return buckets

random.seed(0)
print('no jitter (clients retrying per 100ms bucket):')
for k, v in sorted(simulate(False).items())[:12]:
    print(f'  t={k:>4}s  ' + '#'*v)

print('\nwith jitter:')
for k, v in sorted(simulate(True).items())[:20]:
    print(f'  t={k:>4}s  ' + '#'*v)


### What you should see
- **No jitter**: tall spikes at exactly 0.1s, 0.3s, 0.7s, 1.5s, 3.1s — classic herd.
- **With jitter**: short bars spread across time — much gentler on the service.

### Rules of thumb
- Combine jitter with a **max delay** cap.
- Set a **retry budget** (e.g. 5 attempts max) to avoid retry storms.
- Plus a per-process **circuit breaker** so you stop piling on a dead dep.